# Keeping people cool during a power outage

The outage begins at hour zero. Compare the timing of cooling and outreach, the availability of backup power, and access to transport while crews restore service.

In [ ]:
import json
import tomllib
from pathlib import Path

import napkin
from napkin import store
from napkin.engine import edits_for

here = Path.cwd()
entry = next(p for p in (here / 'model.scn', here / 'scenarios/heat_wave_blackout/model.scn') if p.is_file())
model = napkin.load_scenario(entry)
brief = tomllib.loads(entry.with_name('brief.toml').read_text(encoding='utf-8'))


def completed(runs, edits=None):
    with store.use(store.default_directory()):
        records = napkin.study(model, runs=runs, edits=edits)
    for name, record in records.items():
        if record.stop is not None or record.end != model.base.request.horizon:
            raise ValueError(f'{name} stopped at {record.end:g} h: {record.stopped()}')
    return records


records = completed(['standard', 'early_cooling', 'powered_places', 'delayed_cooling'])

## Service restoration and access to cooling

A service share of 0.5 means half the power supply has returned. Admissions count people who reached a powered cooling site.

In [ ]:
times = (0, 4, 12, 18, 25, 30, 35, 40, 60)
series = {}
for name, record in records.items():
    service, admissions = napkin.read(
        record, model.program, ['supply.share_restored', 'places.admissions'], times
    )
    series[name] = (service.values, admissions.values)
    print(name)
    print('Hour    Supply restored    People admitted to cooling')
    for hour, share, people in zip(times, service.values, admissions.values):
        print(f'{hour:>4}    {share:>14.1%}    {people:>26,.2f}')

In [ ]:
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    print('The table above gives the full series. Install Matplotlib to plot it here.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    for name, (service, admissions) in series.items():
        axes[0].plot(times, service, label=name)
        axes[1].plot(times, admissions, label=name)
    for ax, label in zip(axes, ('Supply share restored', 'People admitted to powered cooling sites')):
        ax.set(xlabel='Hours since the outage', ylabel=label)
    axes[1].legend()
    fig.tight_layout()

## Compare two recovery responses

Choose any pair from the brief below. The table gives each modeled outcome at the end of the study. The change is the comparison run minus the baseline, in the reading's own unit. A cooling-site admission and a device user's hospital admission are separate outcomes.

In [ ]:
pairs = [(row['run'], row['against']) for row in brief['comparison']]
for index, (run, against) in enumerate(pairs):
    print(f'{index}: {run} against {against}')

In [ ]:
def compare(base, other, base_label, other_label):
    paths = (
        'places.admissions',
        'device_admissions',
        'supply.resident_hours',
        'thermal_exposure',
        'casualties',
        'deaths',
    )
    print(f'{base_label} (baseline) and {other_label} (comparison) at {base.end:g} h')
    print(f'{"Reading":<24} {"Unit":<16} {"Baseline":>15} {"Comparison":>15} {"Change":>15}')
    first = napkin.read(base, model.program, paths, [base.end])
    second = napkin.read(other, model.program, paths, [other.end])
    for left, right in zip(first, second, strict=True):
        before, after = left.values[0], right.values[0]
        print(f'{left.path:<24} {left.unit:<16} {before:>15,.4f} {after:>15,.4f} {after - before:>+15,.4f}')


pair_index = 1
run, against = pairs[pair_index]
pair_records = completed([against, run])
compare(pair_records[against], pair_records[run], against, run)

## Inspection timing and recovery

A supplied outage starts the study. Compare a calendar schedule with a follow-up that waits for the previous hold and its repair work to clear. The expedited response uses the same calendar and completes containment earlier. In the acknowledged-stop case the calendar ends when the detection team issues its stop order. Reports after containment add no work. The table also shows the work still waiting and the service already restored.

In [ ]:
inspection_runs = (
    'fixed_inspections',
    'clearance_inspections',
    'rapid_fixed_inspections',
    'acknowledged_stop',
)
inspection_times = (24, 26, 28, 30, 32, 34, 36, 38, 40, 48)
for name, record in completed(inspection_runs).items():
    paths = ['incident.accepted', 'incident.prevented', 'crews.rework_left', 'supply.share_restored']
    rows = napkin.read(record, model.program, paths, inspection_times)
    print(name)
    print('    Hour   Accepted  Prevented    Rework (h)   Supply restored')
    for hour, accepted, blocked, work, share in zip(
        inspection_times, *(row.values for row in rows), strict=True
    ):
        print(f'{hour:8g} {accepted:10.0f} {blocked:10.0f} {work:13.2f} {share:16.1%}')

## How do the responses compare across uncertain inspections?

The published spread varies the interval, hold duration, and added work together. Each response sees the same inputs for a given draw. The difference below is the response minus the ordinary calendar case, calculated for each paired draw before taking the mean and percentiles. With four draws the range describes this exercise only.

In [ ]:
receipt = json.loads((entry.parent / 'results' / 'paired-spreads.json').read_text(encoding='utf-8'))
if receipt['model_hash'] != model.model_hash:
    raise ValueError('The paired results belong to a different model revision.')
spread = next(item for item in receipt['spreads'] if item['id'] == 'inspection_spacing')
for pair in spread['comparisons']:
    print(f'{pair["run"]} against {pair["against"]} ({pair["paired_draws"]} paired draws)')
    print('Reading                      Unit         Mean change       P05       P95')
    for reading in pair['readings']:
        if 'unavailable' in reading:
            print(f'{reading["path"]:<28} {reading["unavailable"]}')
        else:
            delta = reading['difference']
            print(
                f'{reading["path"]:<28} {reading["unit"]:<12} {delta["mean"]:>10,.2f} {delta["p05"]:>9,.2f} {delta["p95"]:>9,.2f}'
            )

## Who reaches cooling?

Select an authored sweep to change backup cooling, private relocation, or another assumption. The comparison runs the selected inputs on the same response and thermal model as its baseline.

In [ ]:
sweeps = {row['id']: row for row in brief['sweep']}
for name, row in sweeps.items():
    print(f'{name:<24} {row["description"]}')

In [ ]:
sweep_id = 'backup_25_private_25'
sweep = sweeps[sweep_id]
baseline = sweep['run']
changed = completed([baseline], edits_for(model, {'set': sweep['set']}))
compare(completed([baseline])[baseline], changed[baseline], baseline, sweep_id)